In [1]:
%load_ext autoreload
%autoreload 2

from collections import deque, Counter

import numpy as np

import matplotlib
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML
import PIL.Image

from dm_control import mjcf
from dm_control import viewer

import torch
import torch.nn as nn
import torch.nn.init as nn_init
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym

from cartpole3d import CartPole3D



In [2]:
cartpole3d_env = CartPole3D(
    nr_movement_dimensions=2,
    force_magnitude=5000,
    physics_steps_per_step=1,
    reset_randomization_magnitude=0.1,
    slide_range=0.5,
    hinge_range=0.8,
    time_limit=3.0,
    step_reward_function=lambda time, action, state: 0, # (time if np.any(np.abs(np.array([1.0, 2.0, 4.0, 6.0, 8.0, 10.0]) - time) < 0.002) else 0) + ((-10) if np.linalg.norm(action) > 0.99 else 0), #-np.linalg.norm(state[:4]) + time / 2,
    out_ouf_range_reward_function=lambda time, action, state: -10 + time * 3,
    time_limit_reward_function=lambda time, action, state: 10000,
)

cartpole3d_env.reset()

for _ in range(200):
    cartpole3d_env.step(np.array([1, 1]))

cartpole3d_env.render()

cartpole3d_env.observation_space

Box([-5.e-01 -5.e-01 -8.e-01 -8.e-01 -1.e+20 -1.e+20 -1.e+20 -1.e+20], [5.e-01 5.e-01 8.e-01 8.e-01 1.e+20 1.e+20 1.e+20 1.e+20], (8,), float32)

In [53]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.set_default_device(device)

eps = np.finfo(np.float32).eps.item()

def step_reward(time, action, state):
    reward = 0

    reward += time 
    
    reward -= np.linalg.norm(state[:4])
    
    return reward

env = CartPole3D(
    nr_movement_dimensions=2,
    force_magnitude=500,
    physics_steps_per_step=1,
    reset_randomization_magnitude=0.1,
    slide_range=1.5,
    hinge_range=0.6,
    time_limit=10.0,
    step_reward_function=step_reward, # (time if np.any(np.abs(np.array([1.0, 2.0, 4.0, 6.0, 8.0, 10.0]) - time) < 0.002) else 0) + ((-10) if np.linalg.norm(action) > 0.99 else 0), #-np.linalg.norm(state[:4]) + time / 2,
    out_ouf_range_reward_function=lambda time, action, state: 0,# -10 + time * 3,
    time_limit_reward_function=lambda time, action, state: 100,
)
# env = gym.make("CartPole-v1")

gamma = 0.75

action_dist_sd = 0.05

minimum_learning_episodes = 25
timestep_discard_limit = 50

class Policy(nn.Module):
    def __init__(self, num_hidden_layers=4, hidden_size=32):
        super(Policy, self).__init__()

        modules = []

        in_dim = {1: 4, 2: 8, 3: 10}[env.nr_movement_dimensions]
        for _ in range(num_hidden_layers):
            linear = nn.Linear(in_dim, hidden_size)
            nn_init.kaiming_uniform_(linear.weight)
            nn_init.zeros_(linear.bias)
            
            norm = nn.LayerNorm(hidden_size)
            activation = nn.ReLU()

            modules.extend([linear, norm, activation])

            in_dim = hidden_size
        
        out_linear = nn.Linear(hidden_size, env.nr_movement_dimensions)
        nn_init.xavier_normal_(out_linear.weight)
        nn_init.zeros_(out_linear.bias)

        activation = nn.Tanh()
        
        modules.extend([out_linear, activation])

        self.fnn = nn.Sequential(*modules)

        self.saved_log_probs = []
        self.rewards = []

    def forward(self, x):
        return self.fnn(x)
    

def select_action(policy, state, pred_log: list[np.ndarray]):
    state = torch.from_numpy(state).float().unsqueeze(0).to(device)
    pred = policy(state)
    pred_log.append(pred.detach().cpu().numpy())
    
    action_dist = torch.distributions.Normal(pred, action_dist_sd)
    action = action_dist.sample()
    policy.saved_log_probs.append(action_dist.log_prob(action))
    
    return action.detach().cpu().numpy().squeeze(0)


def finish_episode(policy, optimizer):
    R = 0
    policy_loss = []
    returns = deque()
    for r in policy.rewards[::-1]:
        R = r + gamma * R
        returns.appendleft(R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + eps)
    for log_prob, R in zip(policy.saved_log_probs, returns):
        policy_loss.append(-log_prob * R)
    optimizer.zero_grad()
    policy_loss = torch.cat(policy_loss).sum()
    policy_loss.backward()
    optimizer.step()
    del policy.rewards[:]
    del policy.saved_log_probs[:]
    

def main():
    best_total_reward = 0

    for i_trial in range(100):
        trial_best_reward = 0
        
        policy = Policy()
        optimizer = optim.Adam(policy.parameters(), lr=1e-3)

        timestep = 0
        for i_episode in range(2000):
            if i_episode > minimum_learning_episodes and timestep < timestep_discard_limit:
                break
            
            state, _ = env.reset()
            ep_reward = 0
            pred_log = []
            info = {}
            for timestep in range(1, 10000):  # Don't infinite loop while learning
                action = select_action(policy, state, pred_log)
                state, reward, done, _, info = env.step(action)
                
                policy.rewards.append(reward)
                ep_reward += reward
                if done:
                    break
    
            finish_episode(policy, optimizer)

            if ep_reward > trial_best_reward:
                trial_best_reward = ep_reward
            
            if ep_reward > best_total_reward:
                best_total_reward = ep_reward
                torch.save(policy, 'best.pt')
            
            if i_episode % 1 == 0:
                def stringify_np_array(arr):
                    return ", ".join([f"{x:> 2.4f}" for x in arr[0]])
                    
                print(f'Episode {i_episode:>4}\t'
                      f'Last reward: {ep_reward:>7.2f} \t '
                      f'Number of Actions: {timestep:>5}       '
                      f'Reason: {info["termination_reason"].split("_")[0]}\t  '
                      f'Pred mean | std | min | max:\t'
                      f'{stringify_np_array(np.mean(pred_log, axis=0))}   |   '
                      f'{stringify_np_array(np.std(pred_log, axis=0))}   |   '
                      f'{stringify_np_array(np.min(pred_log, axis=0))}   |   '
                      f'{stringify_np_array(np.max(pred_log, axis=0))}')
            if False:
                print("Solved! "
                      "the last episode runs to {} time steps!".format(t))
                break
main()

Episode    0	Last reward:  -56.96 	 Number of Actions:   123       Reason: hinge	  Pred mean | std | min | max:	-0.9756,  0.4737   |    0.0303,  0.0817   |   -0.9880,  0.0288   |   -0.7166,  0.7119
Episode    1	Last reward:  -55.70 	 Number of Actions:   124       Reason: hinge	  Pred mean | std | min | max:	-0.9816,  0.3259   |    0.0080,  0.0673   |   -0.9903,  0.1527   |   -0.8954,  0.6133
Episode    2	Last reward:  -57.49 	 Number of Actions:   130       Reason: hinge	  Pred mean | std | min | max:	-0.9792,  0.3614   |    0.0176,  0.1268   |   -0.9868, -0.2477   |   -0.8310,  0.4344
Episode    3	Last reward:  -55.78 	 Number of Actions:   129       Reason: hinge	  Pred mean | std | min | max:	-0.9764,  0.3223   |    0.0416,  0.1521   |   -0.9871, -0.4669   |   -0.5460,  0.5191
Episode    4	Last reward:  -56.48 	 Number of Actions:   128       Reason: hinge	  Pred mean | std | min | max:	-0.9796,  0.2375   |    0.0287,  0.1259   |   -0.9874, -0.2728   |   -0.7160,  0.5466
Episode   

KeyboardInterrupt: 

In [34]:
def main():
    duration = 5   # (Seconds)
    framerate = 30  # (Hz)
    video = []
     
    last_switch = 0
    
    policy = torch.load('best.pt')
    
    physics = env.physics
    
    
    state, _ = env.reset()
    
    trace = []
    
    while physics.data.time < duration:
        action = select_action(policy, state, [])
        trace.append((state, action))
        state, reward, done, _, _ = env.step(action)
    
        if len(video) < physics.data.time * framerate:
            
            pixels = physics.render(width=640, height=480, camera_id=-1)
            video.append(pixels.copy())
        
     
    
    trace_states = [t[0] for t in trace]
    trace_actions = [t[1] for t in trace]
    print(np.mean(trace_actions, axis=0), np.std(trace_actions, axis=0))
    return display_video(video, framerate)

main()

[0.2584968  0.10068764] [0.31311873 0.18484004]


C:\Users\domin\AppData\Local\Temp\ipykernel_7240\2271421933.py:7: MatplotlibDeprecationWarning: Auto-close()ing of figures upon backend switching is deprecated since 3.8 and will be removed two minor releases later.  To suppress this warning, explicitly call plt.close('all') first.
  matplotlib.use(orig_backend)  # Switch back to the original backend.


In [48]:
bucket_size = 25
for i in range(0, len(trace), bucket_size):
    print(f'{i:>5}: \t\t {str(trace_states[i]):<80} {dict(Counter(trace_actions[i:i+bucket_size]))}')

    0: 		 [0.00301361 0.00367415 0.         0.        ]                                    {0: 14, 1: 11}
   25: 		 [ 0.00292876  0.00401671 -0.10202983  0.28135216]                                {1: 11, 0: 14}
   50: 		 [-0.00295657  0.02047702 -0.20547259  0.57582806]                                {0: 11, 1: 14}
   75: 		 [-0.01196818  0.04687255 -0.11103751  0.36473102]                                {1: 14, 0: 11}
  100: 		 [-0.01569649  0.0619232  -0.02066444  0.1907396 ]                                {0: 12, 1: 13}
  125: 		 [-0.01742692  0.07570222 -0.00118233  0.23108436]                                {1: 16, 0: 9}
  150: 		 [-0.01284353  0.0773893   0.21876811 -0.25992532]                                {1: 16, 0: 9}
  175: 		 [ 0.00128048  0.05831053  0.44054853 -0.76694037]                                {1: 13, 0: 12}
  200: 		 [ 0.02666284  0.01235996  0.46714981 -0.79121752]                                {0: 14, 1: 11}
  225: 		 [ 0.04588013 -0.01583977  0.36605822 -

In [16]:
def display_video(frames, framerate=30):
    height, width, _ = frames[0].shape
    dpi = 70
    orig_backend = matplotlib.get_backend()
    matplotlib.use('Agg')  # Switch to headless 'Agg' to inhibit figure rendering.
    fig, ax = plt.subplots(1, 1, figsize=(width / dpi, height / dpi), dpi=dpi)
    matplotlib.use(orig_backend)  # Switch back to the original backend.
    ax.set_axis_off()
    ax.set_aspect('equal')
    ax.set_position([0, 0, 1, 1])
    im = ax.imshow(frames[0])

    def update(frame):
        im.set_data(frame)
        return [im]

    interval = 1000/framerate
    anim = animation.FuncAnimation(fig=fig, func=update, frames=frames,
                                   interval=interval, blit=True, repeat=False)
    return HTML(anim.to_html5_video())

In [ ]:
physics.data.qpos

array([ 0.80046674, -1.047949  ])

In [122]:
np.concatenate([physics.data.qpos, physics.data.qvel])

array([-0.02168953, -0.03073212,  0.38963361, -0.20229654,  0.22114658,
        0.18125404, -1.93815623,  2.08265143])